In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# ==========================================
# 0. SETUP DATA (Simulating Part 2 Structure)
# ==========================================
X, y = make_classification(
    n_samples=1500, n_features=15, n_informative=10, 
    n_classes=2, random_state=42
)
feature_names = [f"feature_{i}" for i in range(X.shape[1])]
X_df = pd.DataFrame(X, columns=feature_names)

X_train, X_test, y_clf_train, y_clf_test = train_test_split(
    X_df, y, test_test_split=0.3, random_state=42, stratify=y
)

# Create pre-scaled variants for standard tree models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled data back to DataFrame to preserve feature column mapping
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_names)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=feature_names)

print("--- Setup Complete. Starting Tasks ---\n")

# ==========================================
# TASK 1: Decision Tree Baseline
# ==========================================
dt_unconstrained = DecisionTreeClassifier(random_state=42)
dt_unconstrained.fit(X_train_scaled_df, y_clf_train)

dt_unconst_train_acc = accuracy_score(y_clf_train, dt_unconstrained.predict(X_train_scaled_df))
dt_unconst_test_acc = accuracy_score(y_clf_test, dt_unconstrained.predict(X_test_scaled_df))

print("Task 1: Unconstrained Decision Tree")
print(f"  Train Accuracy: {dt_unconst_train_acc:.4f}")
print(f"  Test Accuracy:  {dt_unconst_test_acc:.4f}\n")

# ==========================================
# TASK 2: Controlled Decision Tree
# ==========================================
dt_controlled = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
dt_controlled.fit(X_train_scaled_df, y_clf_train)

dt_ctrl_train_acc = accuracy_score(y_clf_train, dt_controlled.predict(X_train_scaled_df))
dt_ctrl_test_acc = accuracy_score(y_clf_test, dt_controlled.predict(X_test_scaled_df))

print("Task 2: Controlled Decision Tree")
print(f"  Train Accuracy: {dt_ctrl_train_acc:.4f}")
print(f"  Test Accuracy:  {dt_ctrl_test_acc:.4f}\n")

# ==========================================
# TASK 3: Gini vs Entropy Comparison
# ==========================================
dt_gini = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
dt_entropy = DecisionTreeClassifier(max_depth=5, criterion='entropy', random_state=42)

dt_gini.fit(X_train_scaled_df, y_clf_train)
dt_entropy.fit(X_train_scaled_df, y_clf_train)

print("Task 3: Criterion Comparison (max_depth=5)")
print(f"  Gini Test Accuracy:    {accuracy_score(y_clf_test, dt_gini.predict(X_test_scaled_df)):.4f}")
print(f"  Entropy Test Accuracy: {accuracy_score(y_clf_test, dt_entropy.predict(X_test_scaled_df)):.4f}\n")

# ==========================================
# TASK 4: Random Forest & Feature Importances
# ==========================================
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train_scaled_df, y_clf_train)

rf_train_acc = accuracy_score(y_clf_train, rf_model.predict(X_train_scaled_df))
rf_test_acc = accuracy_score(y_clf_test, rf_model.predict(X_test_scaled_df))
rf_test_auc = roc_auc_score(y_clf_test, rf_model.predict_proba(X_test_scaled_df)[:, 1])

print("Task 4: Random Forest Performance")
print(f"  Train Accuracy: {rf_train_acc:.4f}")
print(f"  Test Accuracy:  {rf_test_acc:.4f}")
print(f"  Test ROC-AUC:   {rf_test_auc:.4f}\n")

# Extract top features
importances = rf_model.feature_importances_
df_importances = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
df_importances = df_importances.sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("Top 5 Features by Random Forest Importance:")
for idx, row in df_importances.head(5).iterrows():
    print(f"  {idx+1}. {row['Feature']}: {row['Importance']:.4f}")
print("")

# ==========================================
# TASK 4a: Gradient Boosting
# ==========================================
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_model.fit(X_train_scaled_df, y_clf_train)

gb_train_acc = accuracy_score(y_clf_train, gb_model.predict(X_train_scaled_df))
gb_test_acc = accuracy_score(y_clf_test, gb_model.predict(X_test_scaled_df))
gb_test_auc = roc_auc_score(y_clf_test, gb_model.predict_proba(X_test_scaled_df)[:, 1])

print("Task 4a: Gradient Boosting Performance")
print(f"  Train Accuracy: {gb_train_acc:.4f}")
print(f"  Test Accuracy:  {gb_test_acc:.4f}")
print(f"  Test ROC-AUC:   {gb_test_auc:.4f}\n")

# ==========================================
# TASK 4b: Feature Ablation Study
# ==========================================
lowest_5_features = df_importances.tail(5)['Feature'].tolist()
X_train_reduced = X_train_scaled_df.drop(columns=lowest_5_features)
X_test_reduced = X_test_scaled_df.drop(columns=lowest_5_features)

rf_reduced = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_reduced.fit(X_train_reduced, y_clf_train)
rf_reduced_auc = roc_auc_score(y_clf_test, rf_reduced.predict_proba(X_test_reduced)[:, 1])

print("Task 4b: Feature Ablation Study")
print(f"  Removed features: {lowest_5_features}")
print(f"  Full Model ROC-AUC:    {rf_test_auc:.4f}")
print(f"  Reduced Model ROC-AUC: {rf_reduced_auc:.4f}\n")

# ==========================================
# TASK 5: Cross-Validated Comparison
# ==========================================
# Simulating the Logistic Regression baseline from Part 2
lr_model = LogisticRegression(max_iter=1000, random_state=42)

models_to_cv = {
    'Logistic Regression': lr_model,
    'Controlled DT': dt_controlled,
    'Random Forest': rf_model,
    'Gradient Boosting': gb_model
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print("Task 5: 5-Fold Cross-Validation Metrics (ROC-AUC)")
for name, model in models_to_cv.items():
    scores = cross_val_score(model, X_train_scaled_df, y_clf_train, cv=cv_strategy, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = (scores.mean(), scores.std())
    print(f"  {name:20} -> Mean: {scores.mean():.4f} | Std: {scores.std():.4f}")
print("")

# ==========================================
# TASK 6: Hyperparameter Tuning with GridSearchCV
# ==========================================
# Build modern Pipeline using raw untransformed features
tuning_pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}

grid_search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=param_grid,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1
)

# Run grid search on raw inputs
grid_search.fit(X_train, y_clf_train)
best_pipeline = grid_search.best_estimator_

print("Task 6: Hyperparameter Tuning")
print(f"  Best Params: {grid_search.best_params_}")
print(f"  Best CV Score (ROC-AUC): {grid_search.best_score_:.4f}\n")

# ==========================================
# TASK 7: Manual Learning Curve
# ==========================================
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
learning_curve_data = []

print("Task 7: Manual Learning Curve Output")
print(f"{'Training Fraction':<18} | {'Training AUC':<12} | {'Test AUC':<12}")
print("-" * 50)

for f in fractions:
    subset_size = int(f * len(X_train))
    X_train_sub = X_train.iloc[:subset_size]
    y_train_sub = y_clf_train[:subset_size]
    
    # Fit the dynamic pipeline object
    best_pipeline.fit(X_train_sub, y_train_sub)
    
    # Score predictions
    train_auc = roc_auc_score(y_train_sub, best_pipeline.predict_proba(X_train_sub)[:, 1])
    test_auc = roc_auc_score(y_clf_test, best_pipeline.predict_proba(X_test)[:, 1])
    
    learning_curve_data.append((f, train_auc, test_auc))
    print(f"{f:<18.1f} | {train_auc:<12.4f} | {test_auc:<12.4f}")
print("")

# ==========================================
# TASK 8: Serialization & Verification
# ==========================================
# Save the tuned pipeline
joblib.dump(best_pipeline, 'best_model.pkl')
print("Task 8: Saved optimized pipeline as 'best_model.pkl'")

# Complete, self-contained 5+ line validation script
print("\n--- Running Reload & Predict Verification Block ---")
loaded_pipeline = joblib.load('best_model.pkl')
hand_crafted_rows = pd.DataFrame(np.random.randn(2, X_train.shape[1]), columns=feature_names)
predictions = loaded_pipeline.predict(hand_crafted_rows)
pred_probabilities = loaded_pipeline.predict_proba(hand_crafted_rows)[:, 1]

print("Loaded Model Validation Status: SUCCESS")
print(f"Hand-crafted Row Predictions:   {predictions}")
print(f"Prediction Probabilities:       {pred_probabilities}")
print("---------------------------------------------------\n")